# Weather Data Collection for Miami

This notebook pulls historical weather data from multiple sources:
1. NOAA Climate Data Online (CDO) - Primary precipitation data
2. Open-Meteo - Additional weather features
3. MeteoStat - Cross-validation and gap filling

Target: Miami International Airport (25.7617°N, 80.1918°W)

In [5]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timedelta
import json
import time
import os
from dotenv import load_dotenv
from functools import wraps

# Load environment variables from .env file in project root
env_path = os.path.join(os.path.dirname(os.path.abspath('')), '.env')
load_dotenv(dotenv_path=env_path)

# Miami International Airport coordinates
MIAMI_LAT = 25.7617
MIAMI_LON = -80.1918
MIAMI_STATION_ID = "GHCND:USW00012839"

# Date range
START_DATE = "1948-01-01"
END_DATE = datetime.now().strftime("%Y-%m-%d")

# Retry configuration
MAX_RETRIES = 3
RETRY_DELAY = 2  # seconds

def retry_on_failure(max_retries=MAX_RETRIES, delay=RETRY_DELAY):
    """
    Decorator to retry a function on failure.
    
    Args:
        max_retries: Maximum number of retry attempts
        delay: Delay between retries in seconds
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    if attempt < max_retries - 1:
                        print(f"Attempt {attempt + 1}/{max_retries} failed: {str(e)}")
                        print(f"Retrying in {delay} seconds...")
                        time.sleep(delay)
                    else:
                        print(f"All {max_retries} attempts failed.")
            raise last_exception
        return wrapper
    return decorator

## 1. NOAA Climate Data Online (CDO)

**Setup Required:**
1. Get API token from: https://www.ncdc.noaa.gov/cdo-web/token
2. Set environment variable: `export NOAA_TOKEN=your_token_here`

**Rate Limits:** 5 requests/second, 10,000 requests/day

In [6]:
# NOAA CDO API Configuration
NOAA_TOKEN = os.getenv('NOAA_TOKEN')
if not NOAA_TOKEN:
    print("WARNING: NOAA_TOKEN not set. Get token from https://www.ncdc.noaa.gov/cdo-web/token")
    print("Then set it: export NOAA_TOKEN=your_token_here")

NOAA_BASE_URL = "https://www.ncei.noaa.gov/cdo-web/api/v2"
NOAA_HEADERS = {"token": NOAA_TOKEN}

def fetch_noaa_data(station_id, start_date, end_date, dataset_id="GHCND", limit=1000):
    """
    Fetch data from NOAA CDO API with automatic retry on individual request failures.
    
    Args:
        station_id: Station ID (e.g., 'GHCND:USW00012839')
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        dataset_id: Dataset ID (default: GHCND)
        limit: Results per request (max 1000)
    
    Returns:
        DataFrame with weather data
    """
    all_data = []
    offset = 1
    
    while True:
        params = {
            "datasetid": dataset_id,
            "stationid": station_id,
            "startdate": start_date,
            "enddate": end_date,
            "limit": limit,
            "offset": offset,
            "units": "metric"
        }
        
        # Retry individual request with exponential backoff
        success = False
        for attempt in range(MAX_RETRIES):
            try:
                response = requests.get(
                    f"{NOAA_BASE_URL}/data", 
                    headers=NOAA_HEADERS, 
                    params=params, 
                    timeout=30
                )
                
                if response.status_code != 200:
                    error_msg = f"NOAA API error: {response.status_code} - {response.text}"
                    print(error_msg)
                    raise Exception(error_msg)
                
                data = response.json()
                success = True
                break
                
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    wait_time = RETRY_DELAY * (2 ** attempt)  # Exponential backoff
                    print(f"Request failed (attempt {attempt + 1}/{MAX_RETRIES}): {str(e)}")
                    print(f"Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    print(f"All {MAX_RETRIES} attempts failed for offset {offset}")
                    raise
        
        if not success:
            break
        
        if "results" not in data:
            break
        
        all_data.extend(data["results"])
        print(f"Fetched {len(all_data)} records... (offset: {offset})")
        
        # Check if there are more results
        if len(data["results"]) < limit:
            break
        
        offset += limit
        time.sleep(0.5)  # Rate limiting: increased to 2 requests/second to be safe
    
    df = pd.DataFrame(all_data)
    if not df.empty:
        df['date'] = pd.to_datetime(df['date'])
        # Pivot to get datatype as columns
        df_pivot = df.pivot_table(index='date', columns='datatype', values='value', aggfunc='first')
        df_pivot.reset_index(inplace=True)
    else:
        df_pivot = pd.DataFrame()
    
    return df_pivot

In [7]:
# Fetch NOAA data in chunks (1 year at a time to avoid hitting limits)
def fetch_noaa_by_year(station_id, start_year=1948, end_year=None):
    """
    Fetch NOAA data year by year to avoid timeout/limit issues.
    """
    if end_year is None:
        end_year = datetime.now().year
    
    all_dfs = []
    
    for year in range(start_year, end_year + 1):
        start_date = f"{year}-01-01"
        end_date = f"{year}-12-31"
        
        print(f"\nFetching {year}...")
        df_year = fetch_noaa_data(station_id, start_date, end_date)
        
        if not df_year.empty:
            all_dfs.append(df_year)
        
        time.sleep(1.0)  # Increased delay between years
    
    if all_dfs:
        df_combined = pd.concat(all_dfs, ignore_index=True)
        return df_combined
    else:
        return pd.DataFrame()

# Create data directories if they don't exist (relative to project root)
os.makedirs('../data/noaa', exist_ok=True)
os.makedirs('../data/open-meteo', exist_ok=True)
os.makedirs('../data/meteostat', exist_ok=True)

# Example: Fetch recent 2 years (modify as needed)
if NOAA_TOKEN:
    print("Fetching NOAA data for Miami International Airport...")
    df_noaa = fetch_noaa_by_year(MIAMI_STATION_ID, start_year=2023, end_year=2024)
    
    # Save to CSV
    df_noaa.to_csv('../data/noaa/miami_ghcnd.csv', index=False)
    print(f"\nNOAA data saved: {len(df_noaa)} records")
    print(f"Columns: {df_noaa.columns.tolist()}")
    display(df_noaa.head())
else:
    print("Skipping NOAA fetch - token not configured")

Fetching NOAA data for Miami International Airport...

Fetching 2023...
Fetched 1000 records... (offset: 1)
Fetched 2000 records... (offset: 1001)
Fetched 3000 records... (offset: 2001)
Fetched 4000 records... (offset: 3001)
Fetched 5000 records... (offset: 4001)
Fetched 6000 records... (offset: 5001)
Fetched 6806 records... (offset: 6001)

Fetching 2024...
Fetched 1000 records... (offset: 1)
Fetched 2000 records... (offset: 1001)
Fetched 3000 records... (offset: 2001)
Request failed (attempt 1/3): HTTPSConnectionPool(host='www.ncei.noaa.gov', port=443): Read timed out. (read timeout=30)
Retrying in 2 seconds...
Fetched 4000 records... (offset: 3001)
Fetched 5000 records... (offset: 4001)
Fetched 6000 records... (offset: 5001)
Fetched 6766 records... (offset: 6001)

NOAA data saved: 731 records
Columns: ['date', 'ADPT', 'ASLP', 'ASTP', 'AWBT', 'AWND', 'PRCP', 'RHAV', 'RHMN', 'RHMX', 'SNOW', 'SNWD', 'TAVG', 'TMAX', 'TMIN', 'WDF2', 'WDF5', 'WSF2', 'WSF5', 'WT01', 'WT02', 'WT03', 'WT08', 

datatype,date,ADPT,ASLP,ASTP,AWBT,AWND,PRCP,RHAV,RHMN,RHMX,...,WDF2,WDF5,WSF2,WSF5,WT01,WT02,WT03,WT08,WT10,PGTM
0,2023-01-01,211.0,10200.0,10186.0,222.0,2.0,0.0,80.0,57.0,96.0,...,130.0,130.0,5.8,7.2,1.0,NaN,NaN,1.0,NaN,NaN
1,2023-01-02,200.0,10213.0,10200.0,217.0,3.2,0.0,77.0,60.0,96.0,...,110.0,120.0,7.2,8.9,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-01-03,211.0,10207.0,10193.0,228.0,4.7,0.0,80.0,67.0,88.0,...,120.0,160.0,8.1,12.1,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-04,217.0,10203.0,10186.0,228.0,3.8,0.0,82.0,67.0,94.0,...,120.0,110.0,7.2,10.7,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-01-05,206.0,10210.0,10196.0,222.0,2.4,0.0,78.0,55.0,94.0,...,120.0,130.0,6.3,9.4,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Open-Meteo Historical Weather API

**No API key required!**
- Data from 1940 to present (5-day delay)
- Hourly resolution
- Very fast (decades of data in seconds)

In [8]:
def fetch_open_meteo_data(lat, lon, start_date, end_date, hourly_vars=None, daily_vars=None):
    """
    Fetch historical weather data from Open-Meteo with automatic retry on failure.
    
    Args:
        lat: Latitude
        lon: Longitude
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        hourly_vars: List of hourly variables to fetch
        daily_vars: List of daily variables to fetch
    
    Returns:
        Tuple of (hourly_df, daily_df)
    """
    base_url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "timezone": "America/New_York"
    }
    
    if hourly_vars:
        params["hourly"] = ",".join(hourly_vars)
    
    if daily_vars:
        params["daily"] = ",".join(daily_vars)
    
    print(f"Fetching Open-Meteo data from {start_date} to {end_date}...")
    
    # Retry with exponential backoff
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(base_url, params=params, timeout=30)
            
            if response.status_code != 200:
                error_msg = f"Open-Meteo API error: {response.status_code} - {response.text}"
                print(error_msg)
                raise Exception(error_msg)
            
            data = response.json()
            break
            
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                wait_time = RETRY_DELAY * (2 ** attempt)  # Exponential backoff
                print(f"Request failed (attempt {attempt + 1}/{MAX_RETRIES}): {str(e)}")
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"All {MAX_RETRIES} attempts failed.")
                raise
    
    hourly_df = None
    daily_df = None
    
    if "hourly" in data:
        hourly_df = pd.DataFrame(data["hourly"])
        hourly_df['time'] = pd.to_datetime(hourly_df['time'])
    
    if "daily" in data:
        daily_df = pd.DataFrame(data["daily"])
        daily_df['time'] = pd.to_datetime(daily_df['time'])
    
    return hourly_df, daily_df

In [9]:
# Define variables to fetch
hourly_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "rain",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "soil_temperature_0_to_7cm",
    "soil_moisture_0_to_7cm"
]

daily_variables = [
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "rain_sum",
    "precipitation_hours",
    "wind_speed_10m_max",
    "wind_gusts_10m_max",
    "sunshine_duration"
]

# Fetch data (start with recent years, modify dates as needed)
hourly_df, daily_df = fetch_open_meteo_data(
    MIAMI_LAT, 
    MIAMI_LON, 
    "2023-01-01", 
    "2024-12-31",
    hourly_vars=hourly_variables,
    daily_vars=daily_variables
)

if hourly_df is not None:
    hourly_df.to_csv('../data/open-meteo/miami_hourly.csv', index=False)
    print(f"\nOpen-Meteo hourly data saved: {len(hourly_df)} records")
    display(hourly_df.head())

if daily_df is not None:
    daily_df.to_csv('../data/open-meteo/miami_daily.csv', index=False)
    print(f"\nOpen-Meteo daily data saved: {len(daily_df)} records")
    display(daily_df.head())

Fetching Open-Meteo data from 2023-01-01 to 2024-12-31...

Open-Meteo hourly data saved: 17544 records


,time,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,rain,pressure_msl,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,wind_gusts_10m,soil_temperature_0_to_7cm,soil_moisture_0_to_7cm
0,2023-01-01 00:00:00,23.2,89,21.4,0.0,0.0,1019.8,1019.2,0,7.1,156,12.6,22.4,0.170
1,2023-01-01 01:00:00,23.4,88,21.4,0.0,0.0,1019.2,1018.6,9,9.1,146,13.7,22.2,0.170
2,2023-01-01 02:00:00,23.6,87,21.4,0.0,0.0,1018.7,1018.1,14,11.4,152,17.6,22.1,0.171
3,2023-01-01 03:00:00,23.6,87,21.4,0.0,0.0,1018.5,1017.9,29,12.3,159,20.2,22.1,0.171
4,2023-01-01 04:00:00,23.4,89,21.4,0.0,0.0,1018.7,1018.1,97,9.2,169,19.4,22.1,0.172



Open-Meteo daily data saved: 731 records


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,wind_gusts_10m_max,sunshine_duration
0,2023-01-01,26.0,21.2,1.1,1.1,6.0,12.3,20.2,34087.08
1,2023-01-02,26.2,22.5,0.0,0.0,0.0,18.4,29.9,34399.46
2,2023-01-03,26.3,23.2,3.5,3.5,9.0,19.9,36.7,34141.24
3,2023-01-04,26.4,23.3,1.4,1.4,4.0,22.3,40.0,34486.39
4,2023-01-05,27.2,21.4,0.3,0.3,2.0,17.3,31.3,34443.14


## 3. MeteoStat

**Installation:** `pip install meteostat`
- Uses NOAA and other sources
- Simple Python interface
- Returns Pandas DataFrames

In [10]:
# Install meteostat if not already installed
try:
    from meteostat import Point, Daily, Hourly
except ImportError:
    print("Installing meteostat...")
    !pip install meteostat
    from meteostat import Point, Daily, Hourly

Installing meteostat...

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


ImportError: cannot import name 'Daily' from 'meteostat' (/Users/justinbonner/Spring2026/CSC542/GroupProject/.venv/lib/python3.12/site-packages/meteostat/__init__.py)

In [ ]:
# Create location point
miami_point = Point(MIAMI_LAT, MIAMI_LON)

# Fetch daily data with retry logic
@retry_on_failure(max_retries=3, delay=2)
def fetch_meteostat_daily(point, start, end):
    """Fetch MeteoStat daily data with retry on failure."""
    print("Fetching MeteoStat daily data...")
    daily_data = Daily(point, start=start, end=end)
    df = daily_data.fetch()
    if df.empty:
        raise Exception("MeteoStat returned empty dataframe")
    return df

df_meteostat_daily = fetch_meteostat_daily(
    miami_point,
    start=datetime(2023, 1, 1),
    end=datetime(2024, 12, 31)
)

# Save to CSV
df_meteostat_daily.to_csv('../data/meteostat/miami_daily.csv')
print(f"\nMeteoStat daily data saved: {len(df_meteostat_daily)} records")
print(f"Columns: {df_meteostat_daily.columns.tolist()}")
display(df_meteostat_daily.head())

In [ ]:
# Fetch hourly data (optional - can be large for long periods)
@retry_on_failure(max_retries=3, delay=2)
def fetch_meteostat_hourly(point, start, end):
    """Fetch MeteoStat hourly data with retry on failure."""
    print("Fetching MeteoStat hourly data...")
    hourly_data = Hourly(point, start=start, end=end)
    df = hourly_data.fetch()
    if df.empty:
        raise Exception("MeteoStat returned empty dataframe")
    return df

df_meteostat_hourly = fetch_meteostat_hourly(
    miami_point,
    start=datetime(2024, 1, 1),
    end=datetime(2024, 12, 31)
)

# Save to CSV
df_meteostat_hourly.to_csv('../data/meteostat/miami_hourly.csv')
print(f"\nMeteoStat hourly data saved: {len(df_meteostat_hourly)} records")
print(f"Columns: {df_meteostat_hourly.columns.tolist()}")
display(df_meteostat_hourly.head())

## 4. Data Consolidation

Merge data from all sources into a single dataset.

In [ ]:
# Load saved data
try:
    df_noaa = pd.read_csv('../data/noaa/miami_ghcnd.csv', parse_dates=['date'])
    print(f"NOAA data loaded: {len(df_noaa)} records")
except FileNotFoundError:
    df_noaa = pd.DataFrame()
    print("NOAA data not found")

try:
    df_open_meteo = pd.read_csv('../data/open-meteo/miami_daily.csv', parse_dates=['time'])
    df_open_meteo.rename(columns={'time': 'date'}, inplace=True)
    print(f"Open-Meteo data loaded: {len(df_open_meteo)} records")
except FileNotFoundError:
    df_open_meteo = pd.DataFrame()
    print("Open-Meteo data not found")

try:
    df_meteostat = pd.read_csv('../data/meteostat/miami_daily.csv', parse_dates=['time'])
    df_meteostat.rename(columns={'time': 'date'}, inplace=True)
    print(f"MeteoStat data loaded: {len(df_meteostat)} records")
except FileNotFoundError:
    df_meteostat = pd.DataFrame()
    print("MeteoStat data not found")

In [ ]:
# Merge datasets on date
df_consolidated = pd.DataFrame()

# Start with NOAA as base (precipitation target)
if not df_noaa.empty:
    df_consolidated = df_noaa.copy()
    df_consolidated = df_consolidated.add_prefix('noaa_')
    df_consolidated.rename(columns={'noaa_date': 'date'}, inplace=True)

# Merge Open-Meteo
if not df_open_meteo.empty:
    df_open_meteo_prefixed = df_open_meteo.add_prefix('openmeteo_')
    df_open_meteo_prefixed.rename(columns={'openmeteo_date': 'date'}, inplace=True)
    
    if df_consolidated.empty:
        df_consolidated = df_open_meteo_prefixed
    else:
        df_consolidated = pd.merge(df_consolidated, df_open_meteo_prefixed, on='date', how='outer')

# Merge MeteoStat
if not df_meteostat.empty:
    df_meteostat_prefixed = df_meteostat.add_prefix('meteostat_')
    df_meteostat_prefixed.rename(columns={'meteostat_date': 'date'}, inplace=True)
    
    if df_consolidated.empty:
        df_consolidated = df_meteostat_prefixed
    else:
        df_consolidated = pd.merge(df_consolidated, df_meteostat_prefixed, on='date', how='outer')

# Sort by date
if not df_consolidated.empty:
    df_consolidated.sort_values('date', inplace=True)
    df_consolidated.reset_index(drop=True, inplace=True)
    
    # Save consolidated dataset
    df_consolidated.to_csv('../data/miami_consolidated.csv', index=False)
    print(f"\nConsolidated dataset saved: {len(df_consolidated)} records")
    print(f"Date range: {df_consolidated['date'].min()} to {df_consolidated['date'].max()}")
    print(f"\nColumns ({len(df_consolidated.columns)}): {df_consolidated.columns.tolist()}")
    display(df_consolidated.head())
    
    # Data quality summary
    print("\n=== Data Completeness ===")
    print(df_consolidated.isnull().sum())
else:
    print("No data to consolidate")

## 5. Quick Data Exploration

In [ ]:
if not df_consolidated.empty:
    # Basic statistics
    print("=== Dataset Statistics ===")
    display(df_consolidated.describe())
    
    # Check for precipitation columns
    precip_cols = [col for col in df_consolidated.columns if 'prcp' in col.lower() or 'precipitation' in col.lower()]
    print(f"\n=== Precipitation Columns ===")
    print(precip_cols)
    
    if precip_cols:
        display(df_consolidated[['date'] + precip_cols].head(20))

## Next Steps

1. **Extend date range**: Modify `start_year` in NOAA fetch to go back to 1948
2. **Feature engineering**: Create lagged features, rolling averages, seasonal indicators
3. **Handle missing data**: Interpolation, forward fill, or deletion strategies
4. **Target variable**: Define precipitation threshold for classification (e.g., >0mm, >1mm, >10mm)
5. **Add Kalshi data**: When available, add market predictions as features